In [1]:
from pyspark.sql import functions as F
from pyspark.ml.feature import (
    StringIndexer,
    VectorAssembler
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    RegressionEvaluator
)

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Read Cleaned Taxi Data") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()


In [3]:
cleaned_path = "s3a://nyc-taxi/clean"

df = spark.read.parquet(cleaned_path)


In [4]:
df = df.dropna(subset=[
    "fare_amount",
    "trip_distance",
    "payment_type",
    "tip_amount",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime"
])


In [5]:
df = df.withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
       .withColumn("pickup_day", F.dayofweek("tpep_pickup_datetime")) \
       .withColumn("pickup_month", F.month("tpep_pickup_datetime"))

In [6]:
df = df.withColumn(
    "trip_duration",
    (F.unix_timestamp("tpep_dropoff_datetime") -
     F.unix_timestamp("tpep_pickup_datetime")) / 60
)

In [7]:
df = df.withColumn(
    "tip_percent",
    F.when(F.col("fare_amount") > 0,
           F.col("tip_amount") / F.col("fare_amount")
    ).otherwise(0)
)

In [8]:
df = df.withColumn(
    "tipped",
    F.when(F.col("tip_amount") > 0, 1).otherwise(0)
)

In [9]:
payment_indexer = StringIndexer(
    inputCol="payment_type",
    outputCol="payment_type_idx",
    handleInvalid="keep"
)

df = payment_indexer.fit(df).transform(df)

In [10]:
feature_cols = [
    "trip_distance",      # proxy for fare
    "pickup_hour",
    "pickup_day",
    "pickup_month",
    "is_weekend",
    "is_rush_hour",
    "is_night",
    "trip_duration",
    "payment_type_idx",
    "PULocationID",
    "DOLocationID"
]




assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_ml = assembler.transform(df).select(
    "features",
    "tipped",
    "tip_amount",
    "tip_percent"
)

In [11]:
df_model = df_ml.sample(fraction=0.01, seed=42)  # 1%


In [12]:
train_df, test_df = df_model.randomSplit([0.8, 0.2], seed=42)


In [13]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="tipped",
    maxIter=20
)

lr_model = lr.fit(train_df)

In [14]:
class_preds = lr_model.transform(test_df)


In [15]:
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="tipped",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(class_preds)
print("Classification AUC:", auc)


Classification AUC: 0.6442557057552908


In [16]:
from pyspark.sql.functions import rand

train_cls_small = (
    train_df
    .orderBy(rand())
    .limit(200000)
)

train_cls_small.cache()
train_cls_small.count()


200000

In [17]:
from pyspark.ml.classification import GBTClassifier

gbt_cls = GBTClassifier(
    featuresCol="features",
    labelCol="tipped",
    maxDepth=5,
    maxIter=30,
    seed=42
)


In [18]:
gbt_cls_model = gbt_cls.fit(train_cls_small)


In [19]:
cls_preds = gbt_cls_model.transform(test_df)


In [21]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc_eval = BinaryClassificationEvaluator(
    labelCol="tipped",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_eval.evaluate(cls_preds)
print("GBT Classification AUC:", auc)


GBT Classification AUC: 0.6964338679537706


# Feature Importance (GBT Models)

In [20]:
gbt_cls_model.featureImportances


SparseVector(11, {0: 0.1197, 1: 0.1502, 2: 0.0138, 3: 0.0301, 4: 0.0106, 6: 0.0055, 7: 0.1799, 9: 0.292, 10: 0.1981})